In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

# ---------------- Logging helpers ----------------
from datetime import datetime

def log_section(title):
    print("\n" + "=" * 80)
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {title}")
    print("=" * 80)

log_section("STEP 1: INPUT WIDGET VALUES")
print(f"  landing_timestamp : '{landing_timestamp}'")
print(f"  table_name        : '{table_name}'")
print(f"  merge_keys        : '{merge_keys}'")
print(f"  load_type         : '{load_type}'")

empty_inputs = [n for n, v in {"landing_timestamp": landing_timestamp, "table_name": table_name}.items() if not v.strip()]
if empty_inputs:
    print(f"  WARNING: required widget(s) are empty: {empty_inputs}. Downstream steps are likely to fail.")
if not load_type.strip():
    print("  NOTE: load_type is empty -> the write step will default it to MERGE.")
print("  SUCCESS: Widget values loaded.")

In [0]:
from pyspark.sql import functions as F

log_section("STEP 2: SOURCE AND TARGET DETAILS / READ BRONZE DATA")
source_table = f"healthcare_databricks.bronze.{table_name}"
target_table_preview = f"healthcare_databricks.silver.{table_name}"
print(f"  Source table        : {source_table}")
print(f"  Target table        : {target_table_preview}")
print(f"  Source table exists : {spark.catalog.tableExists(source_table)}")
print(f"  Target table exists : {spark.catalog.tableExists(target_table_preview)}")

read_sql = f"SELECT * FROM {source_table} WHERE landing_timestamp = '{landing_timestamp}'"
print("  Equivalent SQL being executed (via DataFrame API):")
print(f"    {read_sql}")

try:
    print("  Reading data from bronze...")
    bronze_df = spark.read.table(f"healthcare_databricks.bronze.{table_name}").filter(f"landing_timestamp='{landing_timestamp}'")
    print(f"  Source columns ({len(bronze_df.columns)}): {bronze_df.columns}")

    total_bronze_count = spark.read.table(source_table).count()
    bronze_count = bronze_df.count()
    print(f"  Total records in source table         : {total_bronze_count}")
    print(f"  Records for landing_timestamp filter  : {bronze_count}")

    if bronze_count == 0:
        print(f"  WARNING: No records found for landing_timestamp='{landing_timestamp}'.")
        latest_ts = (spark.read.table(source_table)
                     .select("landing_timestamp").distinct()
                     .orderBy(F.col("landing_timestamp").desc()).limit(5)
                     .collect())
        if latest_ts:
            print(f"  Latest available landing_timestamp values: {[r['landing_timestamp'] for r in latest_ts]}")
    print("  SUCCESS: Bronze data read completed.")
except Exception as e:
    print(f"  ERROR: Failed to read source table '{source_table}' -> {type(e).__name__}: {e}")
    print("  HINT: Verify the table_name widget and that the bronze table exists.")
    raise

In [0]:
from pyspark.sql import functions as F

log_section("STEP 3: PROCESS DATA (ADD AUDIT COLUMNS)")
print("  Adding audit columns: insert_timestamp = current_timestamp(), update_timestamp = current_timestamp()")
if "insert_timestamp" in bronze_df.columns:
    print("  NOTE: insert_timestamp already exists in bronze data and will be replaced with the current timestamp.")

try:
    updated_df = bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())
    print(f"  Processed columns ({len(updated_df.columns)}): {updated_df.columns}")
    processed_count = updated_df.count()
    print(f"  Records after processing: {processed_count}")
    print("  SUCCESS: Audit columns added.")
except Exception as e:
    print(f"  ERROR: Failed while adding audit columns -> {type(e).__name__}: {e}")
    raise

In [0]:
from delta.tables import DeltaTable

log_section("STEP 4: WRITE TO SILVER")

target_table = f"healthcare_databricks.silver.{table_name}"
load_type_norm = (load_type or 'MERGE').strip().upper()

print(f"  Target table         : {target_table}")
print(f"  load_type (raw)      : '{load_type}'")
if not load_type:
    print("  load_type was empty -> defaulted to MERGE")
print(f"  >>> EXECUTING LOAD TYPE: {load_type_norm} <<<")

# ---------------- Watermark (target state) helper ----------------
def get_target_watermark(tbl):
    """Return max landing_timestamp / max update_timestamp / row count of the target, or None if it doesn't exist."""
    if not spark.catalog.tableExists(tbl):
        return None
    try:
        tdf = spark.table(tbl)
        aggs = [F.count(F.lit(1)).alias("record_count")]
        if "landing_timestamp" in tdf.columns:
            aggs.append(F.max("landing_timestamp").alias("max_landing_timestamp"))
        if "update_timestamp" in tdf.columns:
            aggs.append(F.max("update_timestamp").alias("max_update_timestamp"))
        return tdf.agg(*aggs).first().asDict()
    except Exception as wm_err:
        print(f"  WARNING: Could not compute watermark for {tbl} -> {type(wm_err).__name__}: {wm_err}")
        return None

watermark_before = get_target_watermark(target_table)
print("\n  WATERMARK BEFORE LOAD:")
if watermark_before is None:
    print("    Target table does not exist yet (no watermark).")
else:
    for k, v in watermark_before.items():
        print(f"    {k:<22}: {v}")
print(f"    incoming landing_timestamp: {landing_timestamp}")

try:
    if load_type_norm == "FULL":
        print("\n  FULL LOAD: The target table will be completely overwritten with the source records.")
        print("  Equivalent SQL being executed (via DataFrame API):")
        print(f"    CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <updated_df>")
        # Full load: overwrite the target table
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"  SUCCESS: FULL load completed. {target_table} overwritten.")

    elif load_type_norm == "APPEND":
        print("\n  APPEND LOAD: Source records will be added to the target table; existing rows are untouched.")
        print("  Equivalent SQL being executed (via DataFrame API):")
        print(f"    INSERT INTO {target_table} SELECT * FROM <updated_df>")
        # Append: add the new records to the target table
        updated_df.write.mode("append").saveAsTable(target_table)
        print(f"  SUCCESS: APPEND load completed into {target_table}.")

    elif load_type_norm == "MERGE":
        print("\n  MERGE LOAD: Existing records (matched on merge keys) are UPDATED; new records are INSERTED.")
        # Merge: update existing records and insert new ones based on merge_keys
        keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
        print(f"  Merge keys (raw)    : '{merge_keys}'")
        print(f"  Merge keys (parsed) : {keys}")
        if not keys:
            raise ValueError("merge_keys must be provided (comma-separated) when load_type is MERGE")

        missing = [k for k in keys if k not in updated_df.columns]
        if missing:
            raise ValueError(f"Merge keys not found in source data: {missing}")
        print("  All merge keys exist in the source data.")

        # Avoid multiple source rows matching the same target row
        source_df = updated_df.dropDuplicates(keys)
        pre_dedup_count = updated_df.count()
        post_dedup_count = source_df.count()
        print(f"  Source records before de-duplication on keys : {pre_dedup_count}")
        print(f"  Source records after de-duplication on keys  : {post_dedup_count}")
        print(f"  Duplicate source records removed             : {pre_dedup_count - post_dedup_count}")

        if not spark.catalog.tableExists(target_table):
            print(f"\n  Target table {target_table} does not exist -> first run, creating it with all source records.")
            print("  Equivalent SQL being executed (via DataFrame API):")
            print(f"    CREATE OR REPLACE TABLE {target_table} AS SELECT * FROM <source_df>")
            # First run: create the target table
            source_df.write.mode("overwrite").saveAsTable(target_table)
            print(f"  SUCCESS: Target table created with {post_dedup_count} records.")
        else:
            merge_condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
            # On update keep the original insert_timestamp from the target
            update_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
            insert_set = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

            print(f"\n  Merge condition : {merge_condition}")
            print("  WHEN MATCHED     -> UPDATE all columns except insert_timestamp (original insert time is preserved)")
            print(f"                     Columns updated: {[c for c in source_df.columns if c != 'insert_timestamp']}")
            print("  WHEN NOT MATCHED -> INSERT the full source record")

            merge_sql = (
                f"MERGE INTO {target_table} AS t\n"
                f"    USING <source_df> AS s\n"
                f"    ON {merge_condition}\n"
                f"    WHEN MATCHED THEN UPDATE SET\n        "
                + ",\n        ".join([f"{k} = {v}" for k, v in update_set.items()])
                + "\n    WHEN NOT MATCHED THEN INSERT (" + ", ".join(insert_set.keys()) + ")\n"
                + "        VALUES (" + ", ".join(insert_set.values()) + ")"
            )
            print("  Equivalent SQL being executed (via Delta API):")
            print("    " + merge_sql)

            print("  Executing MERGE...")
            (DeltaTable.forName(spark, target_table).alias("t")
                .merge(source_df.alias("s"), merge_condition)
                .whenMatchedUpdate(set=update_set)
                .whenNotMatchedInsert(values=insert_set)
                .execute())

            # Read MERGE metrics from the latest Delta history entry
            last_op = DeltaTable.forName(spark, target_table).history(1).select("version", "operation", "operationMetrics").first()
            metrics = last_op["operationMetrics"] or {}
            print(f"  Delta version / operation : {last_op['version']} / {last_op['operation']}")
            print(f"  Source rows processed     : {metrics.get('numSourceRows', 'n/a')}")
            print(f"  Rows INSERTED             : {metrics.get('numTargetRowsInserted', 'n/a')}")
            print(f"  Rows UPDATED              : {metrics.get('numTargetRowsUpdated', 'n/a')}")
            print(f"  SUCCESS: MERGE completed into {target_table}.")

    else:
        raise ValueError(f"Unsupported load_type '{load_type}'. Expected one of: FULL, APPEND, MERGE")

except Exception as e:
    print(f"\n  ERROR: {load_type_norm} load into {target_table} failed -> {type(e).__name__}: {e}")
    if isinstance(e, ValueError):
        print("  HINT: Check the load_type and merge_keys widget values.")
    else:
        print("  HINT: Check table permissions, schema compatibility between source and target, and cluster logs.")
    raise

watermark_after = get_target_watermark(target_table)
print("\n  WATERMARK AFTER LOAD:")
if watermark_after is None:
    print("    Target table not available.")
else:
    for k, v in watermark_after.items():
        before_v = watermark_before.get(k) if watermark_before else None
        print(f"    {k:<22}: {before_v} -> {v}")

In [0]:
log_section("STEP 5: SUMMARY")
record_count = updated_df.count()
print(f"  Table processed     : {table_name}")
print(f"  landing_timestamp   : {landing_timestamp}")
print(f"  Load type executed  : {load_type_norm}")
print(f"  Source records      : {record_count}")
print(f"  SUCCESS: bronze -> silver load finished. Exiting notebook with record_count={record_count}")
dbutils.notebook.exit(str(record_count))